In [ ]:
USE DATABASE SNOWFLAKE_LEARNING_DB;
USE SCHEMA SAMMATHEWSON98_LOAD_SAMPLE_DATA_FROM_S3;

In [ ]:
SELECT current_role(), current_warehouse(), current_database(), current_schema();

In [ ]:
SHOW TABLES;

In [ ]:
CREATE OR REPLACE TABLE member_month_spine AS
WITH months AS (
  SELECT DISTINCT DATE_TRUNC('MONTH', date_service) AS as_of_month
  FROM health_claims
),
members AS (
  SELECT DISTINCT patient_id
  FROM health_claims
)
SELECT
  m.patient_id,
  mo.as_of_month
FROM members m
JOIN months mo
  ON EXISTS (
    SELECT 1
    FROM health_claims h
    WHERE h.patient_id = m.patient_id
      AND h.date_service < mo.as_of_month
  );

SELECT * FROM member_month_spine LIMIT 10;


In [ ]:
CREATE OR REPLACE TABLE labels_6m AS
WITH future_spend AS (
  SELECT
    s.patient_id,
    s.as_of_month,
    SUM(h.line_allowed) AS future_allowed_6m
  FROM member_month_spine s
  LEFT JOIN health_claims h
    ON h.patient_id = s.patient_id
   AND h.date_service >= s.as_of_month
   AND h.date_service <  DATEADD(MONTH, 6, s.as_of_month)
  GROUP BY 1,2
),
thresholds AS (
  SELECT
    as_of_month,
    PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY future_allowed_6m) AS p95_allowed
  FROM future_spend
  GROUP BY 1
)
SELECT
  f.patient_id,
  f.as_of_month,
  COALESCE(f.future_allowed_6m, 0) AS future_allowed_6m,
  CASE WHEN COALESCE(f.future_allowed_6m, 0) >= t.p95_allowed THEN 1 ELSE 0 END AS high_cost_next_6m
FROM future_spend f
JOIN thresholds t
  ON f.as_of_month = t.as_of_month;

SELECT * FROM labels_6m WHERE HIGH_COST_NEXT_6M > 0 LIMIT 10;

In [ ]:
CREATE OR REPLACE TABLE features_6m AS
WITH base AS (
  SELECT
    s.patient_id,
    s.as_of_month,
    h.date_service,
    h.line_allowed,
    h.line_charge,
    h.claim_id,
    h.diagnosis_code,
    h.icd10cm_code_description,
    h.cpt,
    h.billing_npi,
    h.location_of_care,
    h.benefit_type
  FROM member_month_spine s
  LEFT JOIN health_claims h
    ON h.patient_id = s.patient_id
   AND h.date_service >= DATEADD(MONTH, -6, s.as_of_month)
   AND h.date_service <  s.as_of_month
),
flagged AS (
  SELECT
    *,
    /* ED proxy */
    CASE WHEN cpt IN ('99281','99282','99283','99284','99285') THEN 1 ELSE 0 END AS is_ed,

    /* High-cost procedure line proxy (tunable threshold) */
    CASE WHEN COALESCE(line_allowed, 0) >= 5000 THEN 1 ELSE 0 END AS is_high_cost_proc,

    /* High-risk / high-cost dx proxy using the description (interpretable) */
    CASE
      WHEN icd10cm_code_description IS NULL THEN 0
      WHEN LOWER(icd10cm_code_description) LIKE ANY (
        '%cancer%', '%malignant%', '%neoplasm%',
        '%heart failure%', '%congestive heart failure%', '%chf%',
        '%stroke%', '%cerebral infar%', '%hemorrhage%',
        '%sepsis%',
        '%renal%', '%kidney%', '%ckd%', '%dialysis%',
        '%copd%', '%respiratory failure%', '%pulmonary embol%',
        '%transplant%',
        '%cirrhosis%', '%liver failure%'
      ) THEN 1
      ELSE 0
    END AS is_high_cost_dx
  FROM base
),

/* Pick ONE representative high-cost procedure CPT per patient-month (top by allowed) */
proc_top AS (
  SELECT
    patient_id,
    as_of_month,
    cpt AS top_high_cost_proc_cpt_6m,
    ROUND(SUM(line_allowed), 2) AS top_high_cost_proc_allowed_6m
  FROM flagged
  WHERE is_high_cost_proc = 1
    AND cpt IS NOT NULL
  GROUP BY 1,2,3
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY patient_id, as_of_month
    ORDER BY SUM(line_allowed) DESC
  ) = 1
)

SELECT
  f.patient_id,
  f.as_of_month,

  /* cost */
  ROUND(COALESCE(SUM(f.line_allowed), 0), 2) AS allowed_6m,
  ROUND(COALESCE(SUM(f.line_charge), 0), 2)  AS paid_6m,

  /* utilization */
  COALESCE(APPROX_COUNT_DISTINCT(f.claim_id), 0) AS claims_6m,
  COALESCE(APPROX_COUNT_DISTINCT(f.date_service), 0) AS service_days_6m,

  /* clinical complexity proxies */
  COALESCE(APPROX_COUNT_DISTINCT(f.diagnosis_code), 0) AS dx_codes_6m,
  COALESCE(APPROX_COUNT_DISTINCT(f.cpt), 0)            AS cpt_codes_6m,
  COALESCE(APPROX_COUNT_DISTINCT(f.billing_npi), 0)    AS providers_6m,

  /* setting + benefit mix */
  COALESCE(SUM(f.is_ed), 0) AS ed_visits_proxy_6m,
  ROUND(COALESCE(SUM(CASE WHEN f.benefit_type = 'PHARMACY' THEN f.line_allowed ELSE 0 END), 0), 2) AS allowed_rx_6m,
  ROUND(COALESCE(SUM(CASE WHEN f.benefit_type <> 'PHARMACY' OR f.benefit_type IS NULL THEN f.line_allowed ELSE 0 END), 0), 2) AS allowed_med_6m,

  /* high-cost dx features */
  COALESCE(SUM(f.is_high_cost_dx), 0) AS high_cost_dx_lines_6m,
  ROUND(COALESCE(SUM(CASE WHEN f.is_high_cost_dx = 1 THEN f.line_allowed ELSE 0 END), 0), 2) AS high_cost_dx_allowed_6m,
  ROUND(
    COALESCE(SUM(CASE WHEN f.is_high_cost_dx = 1 THEN f.line_allowed ELSE 0 END), 0)
    / NULLIF(COALESCE(SUM(f.line_allowed), 0), 0),
    4
  ) AS high_cost_dx_share_allowed_6m,

  /* high-cost procedure features */
  COALESCE(SUM(f.is_high_cost_proc), 0) AS high_cost_proc_lines_6m,
  ROUND(COALESCE(SUM(CASE WHEN f.is_high_cost_proc = 1 THEN f.line_allowed ELSE 0 END), 0), 2) AS high_cost_proc_allowed_6m,
  ROUND(
    COALESCE(SUM(CASE WHEN f.is_high_cost_proc = 1 THEN f.line_allowed ELSE 0 END), 0)
    / NULLIF(COALESCE(SUM(f.line_allowed), 0), 0),
    4
  ) AS high_cost_proc_share_allowed_6m,

  /* NEW: explainability fields about the high-cost procedure */
  CASE WHEN MAX(f.is_high_cost_proc) = 1 THEN 1 ELSE 0 END AS has_high_cost_proc_6m,
  pt.top_high_cost_proc_cpt_6m,
  pt.top_high_cost_proc_allowed_6m,

  /* recency */
  COALESCE(
    DATEDIFF('DAY', MAX(f.date_service), f.as_of_month),
    999
  ) AS days_since_last_claim

FROM flagged f
LEFT JOIN proc_top pt
  ON f.patient_id = pt.patient_id
 AND f.as_of_month = pt.as_of_month
GROUP BY
  f.patient_id,
  f.as_of_month,
  pt.top_high_cost_proc_cpt_6m,
  pt.top_high_cost_proc_allowed_6m;

SELECT *
FROM features_6m
WHERE has_high_cost_proc_6m = 1
LIMIT 25;

In [ ]:
CREATE OR REPLACE TABLE patient_top_dx_6m AS
SELECT
  patient_id,
  DATE_TRUNC('MONTH', date_service) AS as_of_month,
  icd10cm_code_description AS top_dx_description,
  ROUND(SUM(line_allowed), 2) AS allowed
FROM health_claims
GROUP BY 1,2,3
QUALIFY ROW_NUMBER() OVER (
  PARTITION BY patient_id, as_of_month
  ORDER BY SUM(line_allowed) DESC
) = 1;

SELECT * FROM patient_top_dx_6m order by patient_id asc limit 5;

In [ ]:
CREATE OR REPLACE TABLE patient_top_cpt_6m AS
SELECT
  patient_id,
  DATE_TRUNC('MONTH', date_service) AS as_of_month,
  cpt AS cpt,
  ROUND(SUM(line_allowed), 2) AS allowed
FROM health_claims
WHERE cpt IS NOT Null
GROUP BY 1,2,3
QUALIFY ROW_NUMBER() OVER (
  PARTITION BY patient_id, as_of_month
  ORDER BY SUM(line_allowed) DESC
) = 1;

select * from patient_top_cpt_6m order by patient_id asc limit 5;

In [ ]:
CREATE OR REPLACE TABLE risk_model_dataset AS
SELECT
  f.*,
  l.high_cost_next_6m,
  l.future_allowed_6m
FROM features_6m f
JOIN labels_6m l
  ON f.patient_id = l.patient_id
 AND f.as_of_month = l.as_of_month
WHERE f.as_of_month BETWEEN
  (SELECT DATEADD(MONTH, 6, MIN(as_of_month)) FROM member_month_spine)
  AND
  (SELECT DATEADD(MONTH, -6, MAX(as_of_month)) FROM member_month_spine);

  select * from risk_model_dataset limit 10;


In [ ]:
SELECT AVG(high_cost_next_6m) FROM risk_model_dataset;

SELECT
  COUNT_IF(allowed_6m IS NULL),
  COUNT_IF(days_since_last_claim IS NULL)
FROM risk_model_dataset;

SELECT
  AVG(high_cost_next_6m) AS pct_high_cost
FROM risk_model_dataset;

In [ ]:
CREATE OR REPLACE TABLE model_dataset_split AS
SELECT
  *,
  CASE
    WHEN as_of_month < DATEADD(MONTH, -6, (SELECT MAX(as_of_month) FROM risk_model_dataset))
      THEN 'TRAIN'
    WHEN as_of_month < DATEADD(MONTH, -3, (SELECT MAX(as_of_month) FROM risk_model_dataset))
      THEN 'VALID'
    ELSE 'TEST'
  END AS dataset_split
FROM risk_model_dataset;

SELECT
  dataset_split,
  COUNT(*) AS row_count,
  AVG(high_cost_next_6m) AS label_rate
FROM model_dataset_split
GROUP BY 1
ORDER BY 1;

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

import pandas as pd
import numpy as np

# Import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, precision_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# 1) Pull data from Snowflake
df = session.table("MODEL_DATASET_SPLIT").to_pandas()

# 2) Define target and columns to exclude
TARGET = "HIGH_COST_NEXT_6M"
EXCLUDE = {
    "PATIENT_ID",
    "AS_OF_MONTH",
    "DATASET_SPLIT",
    "FUTURE_ALLOWED_6M",
    "TOP_HIGH_COST_PROC_CPT_6M",
    TARGET
}

# Keep only numeric columns
feature_cols = [c for c in df.columns if c not in EXCLUDE]

# Coerce to numeric safely
X = df[feature_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0)
y = df[TARGET].astype(int)

# 3) Time-based splits
train_mask = df["DATASET_SPLIT"] == "TRAIN"
valid_mask = df["DATASET_SPLIT"] == "VALID"
test_mask  = df["DATASET_SPLIT"] == "TEST"

X_train, y_train = X[train_mask], y[train_mask]
X_valid, y_valid = X[valid_mask], y[valid_mask]
X_test,  y_test  = X[test_mask],  y[test_mask]

# 4) Train baseline model
model = Pipeline(steps=[
    ("scaler", StandardScaler(with_mean=False)),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", n_jobs=None))
])
model.fit(X_train, y_train)

# 5) Predict probabilities
valid_proba = model.predict_proba(X_valid)[:, 1] if len(X_valid) else np.array([])
test_proba  = model.predict_proba(X_test)[:, 1]  if len(X_test)  else np.array([])

# 6) Metrics
def precision_at_k(y_true, y_score, k_frac):
    if len(y_true) == 0:
        return None
    k = max(1, int(np.ceil(len(y_true) * k_frac)))
    idx = np.argsort(-y_score)[:k]
    return float(y_true.iloc[idx].mean())

if len(X_valid):
    valid_auc = roc_auc_score(y_valid, valid_proba) if y_valid.nunique() > 1 else None
    valid_p_at_labelrate = precision_at_k(y_valid, pd.Series(valid_proba, index=y_valid.index), y.mean())
else:
    valid_auc = None
    valid_p_at_labelrate = None

if len(X_test):
    test_auc = roc_auc_score(y_test, test_proba) if y_test.nunique() > 1 else None
    test_p_at_labelrate = precision_at_k(y_test, pd.Series(test_proba, index=y_test.index), y.mean())
else:
    test_auc = None
    test_p_at_labelrate = None

print("Label rate (overall):", float(y.mean()))
print("VALID AUC:", valid_auc, "VALID Precision@LabelRate:", valid_p_at_labelrate)
print("TEST  AUC:", test_auc,  "TEST  Precision@LabelRate:", test_p_at_labelrate)

# 7) Write predictions back to Snowflake (TEST + VALID)
pred_df = df.loc[valid_mask | test_mask, ["PATIENT_ID", "AS_OF_MONTH", "DATASET_SPLIT", TARGET]].copy()
pred_df["RISK_SCORE"] = np.nan
pred_df.loc[valid_mask, "RISK_SCORE"] = valid_proba
pred_df.loc[test_mask,  "RISK_SCORE"] = test_proba

# 8) rank within month for operational use
pred_df["AS_OF_MONTH"] = pd.to_datetime(pred_df["AS_OF_MONTH"]).dt.date

pred_df = pred_df.reset_index(drop=True)

session.write_pandas(
    pred_df,
    table_name="PREDICTIONS_BASELINE_LOGREG",
    auto_create_table=True,
    overwrite=True
)

print("Wrote predictions to: PREDICTIONS_BASELINE_LOGREG")

In [ ]:
CREATE OR REPLACE TABLE prediction_explanations AS
SELECT
  p.patient_id,
  p.as_of_month,
  p.dataset_split,
  p.high_cost_next_6m,
  p.risk_score,

  f.allowed_6m,
  f.claims_6m,
  f.dx_codes_6m,
  f.providers_6m,
  f.ed_visits_proxy_6m,
  f.high_cost_dx_share_allowed_6m,
  f.high_cost_proc_share_allowed_6m,

  dx.top_dx_description,
  cpt.cpt AS top_procedure,
  CASE
  WHEN dx.top_dx_description IS NULL
   AND cpt.cpt IS NULL
  THEN 'No dx/proc in lookback window'
  ELSE 'Dx/proc present in lookback'
END AS explanation_context,

FROM predictions_baseline_logreg p
JOIN features_6m f
  ON p.patient_id = f.patient_id
 AND p.as_of_month = f.as_of_month
LEFT JOIN patient_top_dx_6m dx
  ON p.patient_id = dx.patient_id
 AND p.as_of_month = dx.as_of_month
LEFT JOIN patient_top_cpt_6m cpt
  ON p.patient_id = cpt.patient_id
 AND p.as_of_month = cpt.as_of_month;

SELECT *
FROM prediction_explanations
WHERE dataset_split = 'TEST'
ORDER BY risk_score DESC
LIMIT 50;